## This notebook was run in a google colab environment with GPU runtime.

In [1]:
import os

In [2]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [3]:
import wandb
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow import keras as k

## Force W&B Re-login

### Subtask:
Execute `wandb.login(relogin=True)` to force a new login attempt, ensuring any old authentication tokens are cleared and a fresh login is established. This will prompt for the API key again if it's not found in the environment or configuration.


**Reasoning**:
Execute `wandb.login(relogin=True)` as instructed to force a fresh login to Weights & Biases, which will prompt for the API key if needed.



In [4]:
wandb.login(relogin=True)

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sahil-s-mohanty (sahil-s-mohanty-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Reasoning**:
The previous `wandb.login(relogin=True)` step successfully logged the user into Weights & Biases. Now, the next step is to re-run the `FashionMNISTTrainer().train()` code to proceed with the model training, as the authentication issue should be resolved.



In [5]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import numpy as np


# --- custom callbacks ---------------------------------------------------------

class LogLRCallback(k.callbacks.Callback):
    """Log optimizer learning rate each epoch."""
    def on_epoch_end(self, epoch, logs=None):
        opt = self.model.optimizer
        lr = opt.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val}, step=self.model.optimizer.iterations.numpy())

class LogSamplesCallback(k.callbacks.Callback):
    """Log a small table of predictions + images every epoch."""
    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"])
        for i in range(len(self.x)):
            img = self.x[i].squeeze()
            table.add_data(
                wandb.Image(img),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch+1}": table})

class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a confusion matrix from the full validation set each epoch."""
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})

# --- trainer -----------------------------------------------------------------

class FashionMNISTTrainer:
    def __init__(self, project_name="Lab1-visualize-models", run_name="neural_network_plus"):
        self.cfg = dict(
            dropout=0.2,
            layer_1_size=32,
            learn_rate=0.01,   # no decay now
            momentum=0.9,
            epochs=5,
            batch_size=64,
            sample=10000,
        )
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self.labels = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
                       "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
        self._prepare_data()

    def _prepare_data(self):
        (xtr, ytr), (xte, yte) = fashion_mnist.load_data()
        n = self.config.sample
        xtr = xtr[:n].astype("float32")/255.0
        ytr = ytr[:n]
        xte = xte[:n].astype("float32")/255.0
        yte = yte[:n]
        self.X_train = xtr[..., None]
        self.X_test  = xte[..., None]
        self.y_train = to_categorical(ytr)
        self.y_test  = to_categorical(yte)
        self.num_classes = self.y_test.shape[1]

    def _build_model(self):
        inputs = k.Input(shape=(28,28,1))
        x = k.layers.Conv2D(self.config.layer_1_size, (5,5), activation="relu")(inputs)
        x = k.layers.MaxPooling2D((2,2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        x = k.layers.Flatten()(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)
        model = k.Model(inputs, outputs)

        opt = k.optimizers.SGD(
            learning_rate=self.config.learn_rate,  # no decay
            momentum=self.config.momentum,
            nesterov=True,
        )
        model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
        return model

    def _log_model_artifact(self, model):
        # model summary as a text file + the saved model as an artifact
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)
        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/model.h5"
        model.save(model_path)

        art = wandb.Artifact("fashion_mnist_model", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint("checkpoints/model-{epoch:02d}.h5", save_weights_only=False),
            LogLRCallback(),
            LogSamplesCallback(self.X_test, self.y_test, self.labels, max_rows=32),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.labels),
        ]

        model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final/loss": loss, "final/accuracy": acc})

        self._log_model_artifact(model)

        self.run.finish()


FashionMNISTTrainer().train()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5371 - loss: 1.3163

157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.5379 - loss: 1.3138 - val_accuracy: 0.7004 - val_loss: 0.8403
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7758 - loss: 0.6289

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.7759 - loss: 0.6286 - val_accuracy: 0.8087 - val_loss: 0.5393
Epoch 3/5
147/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8221 - loss: 0.5103

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.8220 - loss: 0.5099 - val_accuracy: 0.8105 - val_loss: 0.5261
Epoch 4/5
151/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8408 - loss: 0.4628

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8407 - loss: 0.4627 - val_accuracy: 0.8353 - val_loss: 0.4728
Epoch 5/5
156/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8446 - loss: 0.4434

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.8447 - loss: 0.4432 - val_accuracy: 0.8474 - val_loss: 0.4404


batch/accuracy,▁▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇████████████████████████
batch/batch_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,█▆▅▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▆▆▇█
epoch/val_loss,█▃▂▂▁
+3,...


# Task
Modify the `FashionMNISTTrainer` class by changing `layer_1_size` to 64, `dropout` to 0.3, and `learn_rate` to 0.005. Enhance the `_build_model` method by adding a second `Conv2D` layer with 64 filters followed by a `MaxPooling2D` layer before the `Flatten` layer. Set the `run_name` to 'neural_network_modified' and train the modified model.

## Modify Model and Hyperparameters

### Subtask:
Create a new version of the `FashionMNISTTrainer` class with updated hyperparameters and an enhanced model architecture. Specifically, change `layer_1_size` to 64, `dropout` to 0.3, `learn_rate` to 0.005, and `run_name` to 'neural_network_modified'. Additionally, add a second `Conv2D` layer with 64 filters followed by a `MaxPooling2D` layer in the `_build_model` method before the `Flatten` layer.


**Reasoning**:
I will create a new class, `ModifiedFashionMNISTTrainer`, based on the original `FashionMNISTTrainer`. This class will incorporate the requested hyperparameter changes in its `__init__` method, specifically updating `layer_1_size`, `dropout`, `learn_rate`, and `run_name`.



In [6]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow import keras as k
import os


# --- custom callbacks ---------------------------------------------------------

class LogLRCallback(k.callbacks.Callback):
    """Log optimizer learning rate each epoch."""
    def on_epoch_end(self, epoch, logs=None):
        opt = self.model.optimizer
        lr = opt.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val}, step=self.model.optimizer.iterations.numpy())

class LogSamplesCallback(k.callbacks.Callback):
    """Log a small table of predictions + images every epoch."""
    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"])
        for i in range(len(self.x)):
            img = self.x[i].squeeze()
            table.add_data(
                wandb.Image(img),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch+1}": table})

class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a confusion matrix from the full validation set each epoch."""
    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})

# --- trainer -----------------------------------------------------------------

class ModifiedFashionMNISTTrainer:
    def __init__(self, project_name="Lab1-visualize-models", run_name="neural_network_modified"):
        self.cfg = dict(
            dropout=0.3,
            layer_1_size=64,
            learn_rate=0.005,
            momentum=0.9,
            epochs=5,
            batch_size=64,
            sample=10000,
        )
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self.labels = ["T-shirt/top","Trouser","Pullover","Dress","Coat",
                       "Sandal","Shirt","Sneaker","Bag","Ankle boot"]
        self._prepare_data()

    def _prepare_data(self):
        (xtr, ytr), (xte, yte) = fashion_mnist.load_data()
        n = self.config.sample
        xtr = xtr[:n].astype("float32") / 255.0
        ytr = ytr[:n]
        xte = xte[:n].astype("float32") / 255.0
        yte = yte[:n]
        self.X_train = xtr[..., None]
        self.X_test  = xte[..., None]
        self.y_train = to_categorical(ytr)
        self.y_test  = to_categorical(yte)
        self.num_classes = self.y_test.shape[1]

    def _build_model(self):
        inputs = k.Input(shape=(28,28,1))
        x = k.layers.Conv2D(self.config.layer_1_size, (5,5), activation="relu")(inputs)
        x = k.layers.MaxPooling2D((2,2))(x)
        # Added second Conv2D and MaxPooling2D layers
        x = k.layers.Conv2D(64, (3,3), activation="relu")(x)
        x = k.layers.MaxPooling2D((2,2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        x = k.layers.Flatten()(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)
        model = k.Model(inputs, outputs)

        opt = k.optimizers.SGD(
            learning_rate=self.config.learn_rate,
            momentum=self.config.momentum,
            nesterov=True,
        )
        model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
        return model

    def _log_model_artifact(self, model):
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)
        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/model.h5"
        model.save(model_path)

        art = wandb.Artifact("fashion_mnist_model", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint("checkpoints/model-{epoch:02d}.h5", save_weights_only=False),
            LogLRCallback(),
            LogSamplesCallback(self.X_test, self.y_test, self.labels, max_rows=32),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.labels),
        ]

        model.fit(
            self.X_train, self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final/loss": loss, "final/accuracy": acc})

        self._log_model_artifact(model)

        self.run.finish()

**Reasoning**:
Now that the `ModifiedFashionMNISTTrainer` class is defined, I will instantiate it and call its `train()` method to train the model with the updated hyperparameters and architecture. This will complete the subtask of modifying and training the model.



In [7]:
ModifiedFashionMNISTTrainer().train()

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3482 - loss: 1.8251

157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.3493 - loss: 1.8221 - val_accuracy: 0.7019 - val_loss: 0.8009
Epoch 2/5
147/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7201 - loss: 0.7569

157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.7210 - loss: 0.7545 - val_accuracy: 0.7640 - val_loss: 0.6411
Epoch 3/5
149/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7600 - loss: 0.6418

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7609 - loss: 0.6400 - val_accuracy: 0.7938 - val_loss: 0.5658
Epoch 4/5
143/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7975 - loss: 0.5712

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7980 - loss: 0.5690 - val_accuracy: 0.8072 - val_loss: 0.5450
Epoch 5/5
153/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8018 - loss: 0.5324

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.8020 - loss: 0.5321 - val_accuracy: 0.8211 - val_loss: 0.5105


batch/accuracy,▁▂▂▄▄▄▅▅▇▇▇▇▇▇▇█████████████████████████
batch/batch_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,███▇▇▆▆▆▅▅▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▅▆▇█
epoch/val_loss,█▄▂▂▁
+3,...


**Reasoning**:
Now that the `ModifiedFashionMNISTTrainer` class is defined, I will instantiate it and call its `train()` method to train the model with the updated hyperparameters and architecture. This will complete the subtask of modifying and training the model.



In [8]:
ModifiedFashionMNISTTrainer().train()

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3730 - loss: 1.8223

157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.3739 - loss: 1.8193 - val_accuracy: 0.7158 - val_loss: 0.7772
Epoch 2/5
149/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7083 - loss: 0.7737

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.7096 - loss: 0.7706 - val_accuracy: 0.7769 - val_loss: 0.6296
Epoch 3/5
148/157 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7714 - loss: 0.6196

157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.7716 - loss: 0.6191 - val_accuracy: 0.7904 - val_loss: 0.5859
Epoch 4/5
147/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7901 - loss: 0.5695

157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.7904 - loss: 0.5689 - val_accuracy: 0.7941 - val_loss: 0.5504
Epoch 5/5
145/157 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8125 - loss: 0.5077

157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.8123 - loss: 0.5089 - val_accuracy: 0.8185 - val_loss: 0.5041


batch/accuracy,▁▁▂▂▃▄▄▄▆▆▆▆▇▇▇▇█▇▇▇▇▇▇▇▇███████████████
batch/batch_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,███▇▇▆▅▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▅▆▆█
epoch/val_loss,█▄▃▂▁
+3,...


# Lab Submission: Fashion MNIST Model Training Comparison

This document summarizes two experiments conducted to train a Convolutional Neural Network (CNN) on the Fashion MNIST dataset using Weights & Biases (W&B) for experiment tracking. The goal was to compare the performance of an initial model configuration (`neural_network_plus`) against a modified version (`neural_network_modified`) with altered hyperparameters and an enhanced architecture.

## Experiment 1: `neural_network_plus` (Initial Configuration)

### Hyperparameters:
- `dropout`: 0.2
- `layer_1_size`: 32
- `learn_rate`: 0.01
- `momentum`: 0.9
- `epochs`: 5
- `batch_size`: 64
- `sample`: 10000

### Model Architecture:
- Input Layer: `(28, 28, 1)`
- `Conv2D` layer with 32 filters, (5,5) kernel, 'relu' activation
- `MaxPooling2D` layer with (2,2) pool size
- `Dropout` layer with rate 0.2
- `Flatten` layer
- `Dense` output layer with 10 units (for 10 classes), 'softmax' activation

### Observed Performance:
- **Final Validation Accuracy:** 0.8474
- **Final Validation Loss:** 0.4404

## Experiment 2: `neural_network_modified` (Modified Configuration)

### Hyperparameters:
- `dropout`: 0.3 (changed from 0.2)
- `layer_1_size`: 64 (changed from 32)
- `learn_rate`: 0.005 (changed from 0.01)
- `momentum`: 0.9
- `epochs`: 5
- `batch_size`: 64
- `sample`: 10000

### Model Architecture:
- Input Layer: `(28, 28, 1)`
- `Conv2D` layer with 64 filters, (5,5) kernel, 'relu' activation (changed filter count)
- `MaxPooling2D` layer with (2,2) pool size
- **Added `Conv2D` layer with 64 filters, (3,3) kernel, 'relu' activation**
- **Added `MaxPooling2D` layer with (2,2) pool size**
- `Dropout` layer with rate 0.3 (changed dropout rate)
- `Flatten` layer
- `Dense` output layer with 10 units, 'softmax' activation

### Observed Performance:
- **Final Validation Accuracy:** 0.8185
- **Final Validation Loss:** 0.5041

## Comparison and Analysis:

| Metric                   | `neural_network_plus` | `neural_network_modified` |
| :----------------------- | :-------------------- | :------------------------ |
| **Layer 1 Filters**      | 32                    | 64                        |
| **Second Conv Layer**    | No                    | Yes (64 filters)          |
| **Dropout Rate**         | 0.2                   | 0.3                       |
| **Learning Rate**        | 0.01                  | 0.005                     |
| **Final Val Accuracy**   | 0.8474                | 0.8185                    |
| **Final Val Loss**       | 0.4404                | 0.5041                    |

From the comparison, the initial `neural_network_plus` configuration performed slightly better in terms of both final validation accuracy and loss than the `neural_network_modified` version. The modifications in `neural_network_modified`, which included increasing the `layer_1_size`, adding a second convolutional block, and adjusting `dropout` and `learn_rate`, did not lead to an improved performance in this specific set of experiments. This suggests that the initial simpler architecture with its original hyperparameters was more effective for this dataset and training regime, or that the new combination of hyperparameters and architecture might require further tuning (e.g., more epochs, different learning rate schedules, or regularization adjustments) to unlock potential benefits from the increased complexity.

**Reasoning**:
The training of the `ModifiedFashionMNISTTrainer` model, which incorporates the requested hyperparameter changes and enhanced architecture, has completed successfully as shown by the provided output. This fulfills the task requirements.



In [9]:
print("Model training complete with modified parameters and architecture.")

Model training complete with modified parameters and architecture.


## Final Task

### Subtask:
Review the Weights & Biases dashboard to compare the performance metrics (loss, accuracy, confusion matrix, etc.) of the 'neural_network_modified' run against the previous 'neural_network_plus' run. This will help analyze the impact of the hyperparameter and architectural changes.


## Summary:

### Q&A
The task of modifying the `FashionMNISTTrainer` class with specified hyperparameter and architectural changes, setting the `run_name` to 'neural_network_modified', and training the model was successfully completed.

### Data Analysis Key Findings
*   A new `ModifiedFashionMNISTTrainer` class was created, incorporating the requested changes:
    *   `layer_1_size` was set to 64.
    *   `dropout` was set to 0.3.
    *   `learn_rate` was set to 0.005.
    *   The `run_name` for Weights & Biases was set to 'neural_network_modified'.
    *   The `_build_model` method was enhanced with an additional `Conv2D` layer (64 filters, 3x3 kernel, 'relu' activation) followed by a `MaxPooling2D` layer (2x2 pool size) before the `Flatten` layer.
*   The modified model was trained for 5 epochs.
*   During training, the model showed improving performance on the validation set, achieving a `val_accuracy` of 0.8185 and a `val_loss` of 0.5041 by the end of Epoch 5.
*   Training and validation metrics, along with model artifacts (summary and HDF5 model file), were successfully logged to Weights & Biases.

### Insights or Next Steps
*   The next crucial step is to review the Weights & Biases dashboard to compare the performance of this 'neural_network_modified' run against the previous 'neural_network_plus' run to quantify the impact of the hyperparameter and architectural changes.
